In [64]:
!poetry install
!poetry lock

Installing dependencies from lock file

No dependencies to install or update

Installing the current project: vuala (0.1.0)
If you do not want to install the current project use --no-root.
If you want to use Poetry only for dependency management but not for packaging, you can disable package mode by setting package-mode = false in your pyproject.toml file.
In a future version of Poetry this warning will become an error!
Updating dependencies
Resolving dependencies... (3.5s)Resolving dependencies... (0.2s)

Writing lock file


In [65]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
import pprint

pp = pprint.PrettyPrinter(indent=4, width=80)

openai_organization = "org-a89yiSQTVZ64brAAgexFRIPc"
openai_api_key = os.getenv("OPENAI_API_KEY")


class Request(BaseModel):
    description: str = Field(description="What the additional parameter does")
    url: str = Field(description="URL with parameters candidates")
    expected_status_code: int = Field(description="Expected status code")


class Requests(BaseModel):
    requests: list[Request]


llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    openai_api_key=openai_api_key,
    openai_organization=openai_organization,
).with_structured_output(Requests)

template = """
Give a list of hidden parameters this request can have.

{request}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Give a list of hidden parameters this request can have.

            {request}

            Return list of requests with examples of hidden parameters. Maximum
            10 examples.
            """,
        ),
    ]
)

chain = prompt | llm

In [66]:
request = """
GET /api/products/latest HTTP/2
Host: brokencrystals.com
Accept: application/json, text/plain, */*

HTTP/2 200 OK
Date: Sun, 30 Jun 2024 12:51:21 GMT
Content-Type: application/json; charset=utf-8

[{"id":2,"createdAt":"2024-05-27T08:50:57.000Z","name":"Ruby","category":"Gemstones","photoUrl":"/api/file?path=config/products/crystals/ruby.jpg&type=image/jpg","description":"an intense heart crystal","viewsCount":185078},{"id":8,"createdAt":"2024-05-27T08:50:57.000Z","name":"Bismuth","category":"Gemstones","photoUrl":"/api/file?path=config/products/crystals/bismuth.jpg&type=image/jpg","description":"rainbow","viewsCount":114258},{"id":7,"createdAt":"2024-05-27T08:50:57.000Z","name":"Shattuckite","category":"Jewellery","photoUrl":"/api/file?path=config/products/crystals/shattuckite.jpg&type=image/jpg","description":"mistery","viewsCount":105322}]
"""

requests = chain.invoke({"request": request})

pp.pprint(requests.requests)

[   Request(description='Filter products by category', url='/api/products/latest?category=Gemstones', expected_status_code=200),
    Request(description='Limit the number of products returned', url='/api/products/latest?limit=2', expected_status_code=200),
    Request(description='Sort products by views count in descending order', url='/api/products/latest?sort=viewsCount&order=desc', expected_status_code=200),
    Request(description='Include only products created after a specific date', url='/api/products/latest?createdAfter=2024-05-01', expected_status_code=200),
    Request(description='Include only products with a minimum number of views', url='/api/products/latest?minViews=100000', expected_status_code=200),
    Request(description='Include only products with a specific name', url='/api/products/latest?name=Ruby', expected_status_code=200),
    Request(description='Paginate results, returning the second page', url='/api/products/latest?page=2', expected_status_code=200),
    Requ